# Probe: localization-quality strict DQA-MoX

- created_utc: 2026-05-11T14:00:01+00:00
- target_mAP50: 0.600
- workspace: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27b_probe_localization_uncertainty_r2`
- log: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27b_probe_localization_uncertainty_r2/logs/27b_probe_localization_uncertainty_r2_train.log`
- research_note: `/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/reports/27_research_note_iter_000_27b_probe_localization_uncertainty_r2.md`

## Current Results

| trial | best mAP50 | mAP50:95 |
|---|---:|---:|
| 24a_client_dominant_soft_expand | 0.457000 | 0.258000 |
| 27a_soft_mixture_head_first_40_10 |  |  |
| 27b_localization_uncertainty_strict_then_open |  |  |

## Hypothesis

27b本走行は中間target mAPなしで重すぎるため、同じlocalization-quality仮説を2 roundだけ評価する。ここでwarmupを超えないなら、長い35+15 scheduleには進めずMoE設計を変える。

## Paper Basis

- PseCo: https://arxiv.org/abs/2203.16317
  PseCo argues that classification score alone does not guarantee localization precision; prediction-guided assignment and consistency voting make learning robust to coarse boxes.
- Rethinking Pseudo Labels: https://arxiv.org/abs/2106.00168
  Certainty-aware pseudo labels combine classification and localization quality, dynamically adjust thresholds, and reweight category losses to reduce class imbalance.
- Object-wise Contrastive + Regression Uncertainty: https://arxiv.org/abs/2212.02747
  RUPL-style regression uncertainty separates classification confidence from localization reliability, which matches the observed pseudoGT failure mode in DQA.


In [ ]:
import csv
import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path('/app/Object_Detection')
WORKSPACE = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27b_probe_localization_uncertainty_r2')
LOG_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27b_probe_localization_uncertainty_r2/logs/27b_probe_localization_uncertainty_r2_train.log')
METRICS_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27b_probe_localization_uncertainty_r2/stats/18_client_balanced_single_injection_dqamox_final_metrics.csv')
CMD = ['/opt/venv/bin/python', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/run_scene_daynight_dqa_18_client_balanced_single_injection_dqamox.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27b_probe_localization_uncertainty_r2', '--repair-baseline-rounds', '0', '--source-workspace', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup', '--source-repair-baseline-rounds', '30', '--target-map50', '0.6', '--num-experts', '4', '--top-k', '2', '--router-temperature', '1.15', '--router-balance-weight', '0.02', '--router-entropy-weight', '0.0005', '--dqa-client-balance-stats', '--dqa-client-balance-target', 'median', '--dqa-client-balance-max-scale', '4.0', '--load-bias-strength', '0.25', '--batch-size', '80', '--workers', '8', '--gpus', '2', '--max-images-per-client', '0', '--master-port', '39000', '--evaluate', '--classwise', '--no-eval-plots', '--force', '--warmup-epochs', '50', '--client-limit', '1200', '--client-sampling-ratio', '0.333', '--client-sampling-seed', '270102', '--phase1-rounds', '2', '--phase2-rounds', '0', '--phase1-train-scope', 'neck_head', '--phase1-repair-train-scope', 'neck_head', '--phase1-client-epochs', '1', '--phase1-client-lr', '0.00028', '--phase1-source-repeat', '1', '--phase1-pseudo-repeat', '2', '--phase1-loss-box', '0.00045', '--phase2-train-scope', 'all', '--phase2-repair-train-scope', 'all', '--phase2-client-epochs', '1', '--phase2-client-lr', '0.00004', '--phase2-source-repeat', '1', '--phase2-pseudo-repeat', '1', '--phase2-loss-box', '0.00005', '--server-repair-epochs', '1', '--server-repair-lr', '0.00018', '--server-repair-loss-box', '0.010', '--dqa-server-anchor', '0.16', '--dqa-min-server-alpha', '0.10', '--dqa-residual-blend', '0.00', '--late-dqa-server-anchor', '0.08', '--late-dqa-min-server-alpha', '0.03', '--late-dqa-residual-blend', '0.00', '--curriculum-start-round', '3', '--expert-keep-fraction', '0.78', '--expert-max-class-fraction', '0.30', '--actual-max-class-fraction', '0.44', '--late-expert-keep-fraction', '0.92', '--late-expert-max-class-fraction', '0.40', '--late-actual-max-class-fraction', '0.58', '--min-score', '0.20', '--min-stability', '0.68', '--late-min-score', '0.14', '--late-min-stability', '0.48', '--max-boxes-per-image', '8', '--skip-warmup-training', '--warmup-checkpoint', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/08_full_latent_dqamox_from_warmup/checkpoints/round000_latent_dqamox_warmup.pt']

WORKSPACE.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "stats").mkdir(parents=True, exist_ok=True)
(WORKSPACE / "stats" / "27_notebook_command.json").write_text(
    json.dumps({"command": CMD}, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(" ".join(CMD))
with LOG_PATH.open("w", encoding="utf-8") as log:
    proc = subprocess.run(CMD, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT, check=False)
print("returncode", proc.returncode)
print("log", LOG_PATH)
if proc.returncode != 0:
    raise SystemExit(proc.returncode)


In [ ]:
import csv
from pathlib import Path

METRICS_PATH = Path('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/27_research_notebook_until_060/27b_probe_localization_uncertainty_r2/stats/18_client_balanced_single_injection_dqamox_final_metrics.csv')
rows = list(csv.DictReader(METRICS_PATH.open(encoding="utf-8"))) if METRICS_PATH.exists() else []
for row in rows:
    print(row)
